# 04 finetune

Step 4 of the guide: fine-tune the pretrained decoder to say whether one transaction is fraud, and train the same architecture from scratch as the control. The difference between the two is "what pretraining bought".

```
one example  =  <BOS> | tx[i-12] ... tx[i-1] | tx[i] | <PAD> ...        1 + 13 x 7 = 92 tokens
                        12 previous transactions     ^ the row judged; the representation of its last
                        of the same card (fewer       token is what the classifier reads (last-token pooling,
                        if the card is young)         GPT2ForSequenceClassification with pad_token_id set)
```

| setting | value | guide |
|---|---|---|
| history H | 12 transactions | default; step 6 varies 0 / 5 / 12 / 32 |
| train examples | every train-period fraud + 5% of train-period normals | WF keeps 1%; small data, so 5% |
| val and test | all rows, not sampled | otherwise precision-type metrics are inflated |
| class weight | none | sampling or weights, not both |
| learning rate, epochs | 5e-5, 3 epochs | one order below pretraining |
| checkpoint | highest validation PR-AUC | eval and save every 500 steps on Drive |
| control | same config, random init | "how much did pretraining bring" |
| seeds | 0, 1, 2 per setting | mean and sd reported |

In [ ]:
import os, sys, glob, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
for f in glob.glob("notebooks/*.ipynb"):          # new notebooks from the repo appear in Drive
    dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok")

## What trains on what

- Targets: train examples are train rows; val and test examples are their own rows, with the true 0.5% fraud rate.
- History: the 12 rows before the target on the same card, whatever split they fall in. A test row seeing its card's train-period rows is the allowed direction (a production scorer sees the past). Nothing ever looks forward, and no example crosses a card (`tests/test_finetune.py`).
- Labels never enter the tokens. The history rows' `is_fraud` is not used anywhere; `test_changing_the_label_changes_y_but_never_the_tokens` flips every label and checks the token matrix is unchanged.
- Beware one Sparkov quirk when reading the decoded sequences: every merchant in the dataset is named `fraud_<Name>`, so `MER_fraud_...` tokens appear on normal and fraud rows alike. It is a naming artefact of the generator, not label information.
- The tokenizer is loaded from `pretrain/best/tokenizer.json`, the copy that shipped with the checkpoint, so weights and vocabulary cannot be mismatched.

## Stage 4a: build the examples

In [ ]:
import json, time, os, glob, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from tfm.tokenizer import TransactionTokenizer
from tfm.finetune import build_examples
from tfm.metrics import evaluate_scores, mean_sd, correct_for_sampling

P, PRE, RES = f"{ROOT}/data/processed", f"{ROOT}/pretrain/best", f"{ROOT}/results"
df = pd.read_parquet(f"{P}/transactions.parquet")
tok = TransactionTokenizer.load(f"{PRE}/tokenizer.json")          # the vocabulary that ships with the checkpoint
ids_all = tok.encode(df)
card, split, fraud = df["card_id"].to_numpy(), df["split"].to_numpy(), df["is_fraud"].to_numpy()
PAD, BOS, T = tok.vocab["<PAD>"], tok.vocab["<BOS>"], tok.tokens_per_tx
H, KEEP, ALERT = 12, 0.05, 0.006

ex = {"train": build_examples(ids_all, card, split, fraud, "train", H, PAD, BOS, neg_keep_rate=KEEP, seed=0),
      "val":   build_examples(ids_all, card, split, fraud, "val",   H, PAD, BOS),
      "test":  build_examples(ids_all, card, split, fraud, "test",  H, PAD, BOS)}
rows = []
for s, (X, y, idx) in ex.items():
    rows.append({"split": s, "rows in split": int((split == s).sum()), "examples": len(y), "fraud": int(y.sum()), "fraud rate": float(y.mean()),
                 "normals sampled": "5% (seed 0)" if s == "train" else "no", "seq len": X.shape[1], "mean real tokens": float((X != PAD).sum(1).mean()),
                 "examples with full history": float(((X != PAD).sum(1) == X.shape[1]).mean())})
display(pd.DataFrame(rows).style.format({"fraud rate": "{:.2%}", "mean real tokens": "{:.1f}", "examples with full history": "{:.1%}", "rows in split": "{:,}", "examples": "{:,}", "fraud": "{:,}"}))
print(f"sequence length {ex['train'][0].shape[1]} = 1 (<BOS>) + ({H} history + 1 current) x {T} tokens; dtype {ex['train'][0].dtype}")
print("the train sample is rebuilt with each run's seed in Stage 4d, so the three seeds also vary the sampled normals")

## Stage 4b: one example, decoded, next to the raw rows

The first fraud in the test period: its card's 12 previous rows and the row itself, as raw columns (with `is_fraud`, for the reader only) and as the tokens the model gets. Then a machine check on 2,000 random examples per split: the current transaction's tokens sit right before the first pad and equal the tokenizer's encoding of that row, and the history slots equal the encodings of the same card's preceding rows.

In [ ]:
X, y, idx = ex["test"]
k = int(np.flatnonzero(y == 1)[0]); i = int(idx[k])
pos = df.groupby("card_id").cumcount().to_numpy()
h = int(min(pos[i], H))
raw = df.iloc[i - h: i + 1][["card_id", "split", "ts", "category", "merchant", "amt", "dist_km", "is_fraud"]].copy()
raw["tokens"] = [" ".join(tok.decode(X[k, 1 + s * T: 1 + (s + 1) * T])) for s in range(h + 1)]
display(raw.style.format({"amt": "{:.2f}", "dist_km": "{:.1f}"}))
print(f"row {i}: {h} history rows, {(X[k] != PAD).sum()} real tokens, label y = {int(y[k])}; the last real token is {tok.decode([X[k][(X[k] != PAD).sum() - 1]])[0]} of the judged row")

rng = np.random.default_rng(0); bad = 0; checked = 0
for s, (Xs, ys, ids_s) in ex.items():
    for k2 in rng.choice(len(ys), min(2000, len(ys)), replace=False):
        i2 = int(ids_s[k2]); h2 = int(min(pos[i2], H)); seq = Xs[k2]
        n_real = int((seq != PAD).sum())
        ok = n_real == 1 + (h2 + 1) * T and seq[0] == BOS
        ok &= np.array_equal(seq[1 + h2 * T: 1 + (h2 + 1) * T], ids_all[i2])                    # current row last
        ok &= np.array_equal(seq[1: 1 + h2 * T].reshape(h2, T), ids_all[i2 - h2: i2]) if h2 else True   # history = preceding rows
        ok &= (card[i2 - h2: i2 + 1] == card[i2]).all() and int(ys[k2]) == int(fraud[i2])
        bad += not ok; checked += 1
print(f"machine check on {checked:,} random examples: {checked - bad:,} consistent, {bad} inconsistent")
assert bad == 0

## Stage 4c: the two models

`pretrained`: the body from `pretrain/best`, a new 2-class head. `scratch`: the same GPT-2 config with random weights, also a new head. Everything else (examples, seeds, schedule, selection) is identical, so the gap is attributable to pretraining alone.

In [ ]:
import torch
from tfm.finetune import load_classifier
m = load_classifier("pretrained", PRE, len(tok.vocab), PAD, BOS)
print(f"{type(m).__name__}: {sum(p.numel() for p in m.parameters()) / 1e6:.2f}M parameters; body from {PRE}; head 'score' {tuple(m.score.weight.shape)} newly initialised; pad_token_id {m.config.pad_token_id} (last non-pad token is pooled)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: Runtime -> Change runtime type -> T4")
del m
!PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -1

## Stage 4d: six runs (2 settings x 3 seeds), each resumable

Each run has its own Drive folder `finetune/<init>_H12/seed<k>/` and its own result file `results/04_<init>_H12_seed<k>.json`. A run whose result file exists is skipped, so if the session dies this cell is simply rerun: finished runs are kept, the interrupted one resumes from its newest complete checkpoint, the rest start. To redo a run on purpose, delete its json. Test scores are saved as `test_scores.npy` next to the checkpoints for step 6 (error analysis, bootstrap).

Order: pretrained seeds 0, 1, 2, then scratch 0, 1, 2, so the main result is complete first if time runs out.

In [ ]:
from pathlib import Path
from transformers import Trainer, PrinterCallback
from tfm.checkpointing import find_latest_checkpoint, verify_checkpoint
from tfm.finetune import ClassificationDataset, finetune_args, compute_metrics_fn, softmax_fraud_prob, best_eval
from tfm.pretrain import print_progress_callback

Xv, yv, iv = ex["val"]; Xt, yt, it = ex["test"]
val_ds, test_ds = ClassificationDataset(Xv, yv, PAD), ClassificationDataset(Xt, yt, PAD)
RUNS = [(init, seed) for init in ("pretrained", "scratch") for seed in (0, 1, 2)]
for init, seed in RUNS:
    name = f"04_{init}_H{H}_seed{seed}"; out = f"{ROOT}/finetune/{init}_H{H}/seed{seed}"; res = f"{RES}/{name}.json"
    if os.path.exists(res):
        print(f"{name}: done earlier, skipped"); continue
    print(f"\n==== {name} ====")
    Xtr, ytr, itr = build_examples(ids_all, card, split, fraud, "train", H, PAD, BOS, neg_keep_rate=KEEP, seed=seed)
    model = load_classifier(init, PRE, len(tok.vocab), PAD, BOS)
    trainer = Trainer(model=model, args=finetune_args(out, seed=seed), train_dataset=ClassificationDataset(Xtr, ytr, PAD),
                      eval_dataset=val_ds, compute_metrics=compute_metrics_fn(card[iv], ALERT), callbacks=[print_progress_callback()])
    trainer.remove_callback(PrinterCallback)
    last = find_latest_checkpoint(out)
    print(f"{len(ytr):,} train examples ({int(ytr.sum()):,} fraud)   resuming from: {last.name if last else 'start'}")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=str(last) if last else None)
    for p in sorted(Path(out).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])):
        print(f"  {p.name:18s}", verify_checkpoint(p) or "ok")
    best = best_eval(trainer.state.log_history)                              # the checkpoint load_best_model_at_end restored
    trainer.compute_metrics = None                                            # test has other card ids; metrics computed below
    p_test = softmax_fraud_prob(trainer.predict(test_ds).predictions)
    np.save(f"{out}/test_scores.npy", p_test.astype(np.float32))
    test_m = evaluate_scores(yt, p_test, card[it], ALERT)
    summary = {"name": name, "init": init, "history": H, "seed": seed, "neg_keep_rate": KEEP, "alert_rate": ALERT,
               "n_train_examples": int(len(ytr)), "n_train_fraud": int(ytr.sum()), "best_step": best["step"],
               "val_at_best": {k[len("eval_"):]: v for k, v in best.items() if k.startswith("eval_")}, "test": test_m,
               "test_fraud_rate": float(yt.mean()), "mean_prob_test_raw": float(p_test.mean()),
               "mean_prob_test_corrected": float(correct_for_sampling(p_test, KEEP).mean()), "wall_min": round((time.time() - t0) / 60, 1)}
    with open(res, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"{name}: best step {best['step']}  val PR-AUC {best['eval_pr_auc']:.4f}  |  test PR-AUC {test_m['pr_auc']:.4f}  "
          f"recall@{ALERT:.1%} {test_m['recall_at_alert']:.3f}  card discovery {test_m['card_discovery']:.3f}  FP ratio {test_m['fp_ratio']:.2f}  |  {summary['wall_min']} min")
    del trainer, model; torch.cuda.empty_cache()

## Stage 4e: results, mean and sd over seeds

One row per run, then the aggregate per setting on the test set, then the pretraining gain (pretrained minus scratch) per metric. The last line is the calibration note the guide asks for: the mean predicted probability on test is inflated by the 5% sampling; `correct_for_sampling` brings it back near the true fraud rate without changing any ranking metric.

In [ ]:
runs = [json.load(open(f)) for f in sorted(glob.glob(f"{RES}/04_*_seed*.json"))]
KEYS = ["pr_auc", "roc_auc", "recall_at_alert", "card_discovery", "fp_ratio"]
flat = pd.DataFrame([{"init": r["init"], "seed": r["seed"], "best_step": r["best_step"], "val PR-AUC": r["val_at_best"]["pr_auc"],
                      **{f"test {k}": r["test"][k] for k in KEYS}, "min": r["wall_min"]} for r in runs])
display(flat.style.format(precision=4))
agg = {init: mean_sd([r["test"] for r in runs if r["init"] == init]) for init in ("pretrained", "scratch") if any(r["init"] == init for r in runs)}
tbl = pd.DataFrame({f"{init} {stat}": {k: agg[init][k][stat] for k in KEYS} for init in agg for stat in ("mean", "sd")})
display(tbl.style.format("{:.4f}"))
gain = {}
if {"pretrained", "scratch"} <= set(agg):
    for k in KEYS:
        gain[k] = agg["pretrained"][k]["mean"] - agg["scratch"][k]["mean"]
        print(f"{k:16s} pretrained {agg['pretrained'][k]['mean']:.4f} +- {agg['pretrained'][k]['sd']:.4f}   scratch {agg['scratch'][k]['mean']:.4f} +- {agg['scratch'][k]['sd']:.4f}   gain {gain[k]:+.4f}")
r0 = runs[0]
print(f"\ncalibration ({r0['name']}): test fraud rate {r0['test_fraud_rate']:.3%}, mean predicted prob raw {r0['mean_prob_test_raw']:.3%}, after sampling correction {r0['mean_prob_test_corrected']:.3%}")

## Stage 4f: precision-recall curves and the step-4 json

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
yt = ex["test"][1]
fig, ax = plt.subplots(figsize=(6, 4.5))
for r in runs:
    if r["seed"] != 0: continue
    p = np.load(f"{ROOT}/finetune/{r['init']}_H{H}/seed0/test_scores.npy")
    prec, rec, _ = precision_recall_curve(yt, p)
    ax.plot(rec, prec, label=f"{r['init']} (PR-AUC {r['test']['pr_auc']:.3f})")
ax.axhline(float(yt.mean()), ls="--", c="gray", label=f"fraud rate {yt.mean():.2%}")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title("test set, seed 0"); ax.legend(); plt.tight_layout()
fig.savefig(f"{RES}/04_finetune_pr.png", dpi=120); plt.show()

out = {"history": H, "neg_keep_rate": KEEP, "alert_rate": ALERT, "n_test": int(len(yt)), "n_test_fraud": int(yt.sum()),
       "runs": runs, "aggregate_test": agg, "pretraining_gain": gain}
with open(f"{RES}/04_finetune.json", "w") as f:
    json.dump(out, f, indent=2)
print("saved 04_finetune.json:", list(out))

## What to read off the numbers

- `gain` on PR-AUC and recall at 0.6% alerts is the headline: pretraining's value at this scale. A gain inside one sd is "no measurable gain", and that is a legitimate finding to report, not a failure.
- `card discovery` vs `recall_at_alert`: discovery can be high while recall is modest when frauds come in bursts on a few cards (fraud_burst_share was 0.90 in step 1): catching one fraud per card is enough to find the card.
- `fp_ratio` is what an operations team feels; WF reported it next to recall for the same reason.
- `best_step` far below the last step means 3 epochs was already too many for this data; the PR-AUC on val chose earlier.
- Scores near 1.0 everywhere are the Sparkov warning from the guide: rule-generated fraud is easy. Then the fixed-alert-rate differences and the step-6 ablation trends carry the report, not the absolute numbers.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
    cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
    return cells(a) == cells(b)
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
    repo = f"notebooks/{os.path.basename(f)}"
    if not os.path.exists(repo) or same_code(f, repo):
        shutil.copy(f, repo)
    else:
        print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
!git log --oneline | head -1

In [ ]:
drive.flush_and_unmount()